# notebooks/02_feature_engineering.ipynb

In [97]:
import sys, os

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath(os.path.join('..', 'scripts')))
from data_loader import create_panel_data

## Load Panel Data

In [98]:
panel_df = create_panel_data(frequency='5_min')

print('Data Head: ')
print(panel_df.head())
print('\nData Tail: ')
print(panel_df.tail())

/Users/xizuo/MLProject/scripts/data_loader.py:61: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  stacked = df.stack(dropna=False)
/Users/xizuo/MLProject/scripts/data_loader.py:61: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  stacked = df.stack(dropna=False)
/Users/xizuo/MLProject/scripts/data_loader.py:61: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  stacked = 

Data Head: 
                        rv       bpv      good       bad          rq
Date       Stock                                                    
2003-01-02 AAPL   6.493909  3.771960  5.102315  1.391595  152.729402
           AMGN   5.177506  3.635898  3.362921  1.814585   46.168213
           AMZN   9.886836  8.018926  6.074276  3.812559  200.751309
           AXP    4.448244  4.122573  3.337253  1.110991  114.695371
           BA     7.469396  7.016747  5.107386  2.362010  150.112379

Data Tail: 
                        rv       bpv      good       bad        rq
Date       Stock                                                  
2024-03-28 TRV    0.501900  0.450509  0.225921  0.275979  0.517683
           UNH    0.774552  0.762729  0.406755  0.367797  0.696658
           V      0.627872  0.445852  0.329977  0.297895  1.160977
           VZ     0.783853  0.706211  0.493097  0.290755  1.847938
           WMT    0.359440  0.336123  0.106481  0.252960  0.150975


/Users/xizuo/MLProject/scripts/data_loader.py:61: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  stacked = df.stack(dropna=False)


## Integrate VIX Data

In [99]:
START_DATE = '2003-01-02'
END_DATE = '2024-03-28'

vix_path = os.path.join('..', 'data', 'VIX_History.csv')
vix_data = pd.read_csv(vix_path)

vix_data['Date'] = pd.to_datetime(vix_data['DATE'], format='%m/%d/%Y')
vix_data = vix_data.set_index('Date')
vix = vix_data[['CLOSE']].rename(columns={'CLOSE': 'vix'})
vix = vix.loc[START_DATE:END_DATE]

panel_vix = panel_df.join(vix, on='Date')
print('Panel with VIX: ')
print(panel_vix.head())

Panel with VIX: 
                        rv       bpv      good       bad          rq    vix
Date       Stock                                                           
2003-01-02 AAPL   6.493909  3.771960  5.102315  1.391595  152.729402  25.39
           AMGN   5.177506  3.635898  3.362921  1.814585   46.168213  25.39
           AMZN   9.886836  8.018926  6.074276  3.812559  200.751309  25.39
           AXP    4.448244  4.122573  3.337253  1.110991  114.695371  25.39
           BA     7.469396  7.016747  5.107386  2.362010  150.112379  25.39


## Define Target Variable (Y_reg)

In [100]:
df = panel_vix.sort_index()

df['Y_reg'] = df.groupby('Stock')['rv'].shift(-1)

print('Example for AAPL: ')
display(df.loc[pd.IndexSlice[:, 'AAPL'], :].tail()[['rv', 'Y_reg']])

Example for AAPL: 


,,rv,Y_reg
Date,Stock,,
2024-03-22,AAPL,1.216548,0.682342
2024-03-25,AAPL,0.682342,0.425990
2024-03-26,AAPL,0.425990,0.959378
2024-03-27,AAPL,0.959378,0.563307
2024-03-28,AAPL,0.563307,NaN


In [ ]:
# feature engineering

grp = df.groupby(level='Stock', group_keys=False)

# good / bad: lag1
df['good_lag1'] = grp['good'].shift(1)
df['bad_lag1']  = grp['bad'].shift(1)

# rv: lag1-5
for k in [1, 2, 3, 4, 5]:
    df[f'rv_lag{k}'] = grp['rv'].shift(k)

# rv: 2/3/5 daily move average 
for w in [2, 3, 5]:
    df[f'rv_ma{w}'] = grp['rv'].shift(1).rolling(w, min_periods=w).mean()

# rv: log / sqrt
df['rv_log']  = np.log(df['rv'].clip(lower=1e-12))
df['rv_sqrt'] = np.sqrt(df['rv'].clip(lower=0))

# vix: lag1-5
for k in [1, 2, 3, 4, 5]:
    df[f'vix_lag{k}'] = grp['vix'].shift(k)


all_features = (
    [f'rv_lag{k}' for k in [1,2,3,4,5]] +
    [f'rv_ma{w}' for w in [2,3,5]] +
    ['rv_log', 'rv_sqrt', 'good_lag1', 'bad_lag1'] +
    [f'vix_lag{k}' for k in [1,2,3,4,5]]
)
print("feature list：\n")
for i, feat in enumerate(all_features, start=1):
    print(f"{i:2d}. {feat}")





feature list：

 1. rv_lag1
 2. rv_lag2
 3. rv_lag3
 4. rv_lag4
 5. rv_lag5
 6. rv_ma2
 7. rv_ma3
 8. rv_ma5
 9. rv_log
10. rv_sqrt
11. good_lag1
12. bad_lag1
13. vix_lag1
14. vix_lag2
15. vix_lag3
16. vix_lag4
17. vix_lag5


In [105]:
# choose feature for HAR
selected_feats = ['rv_ma5','bpv','good_lag1','bad_lag1','rq','vix']


from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error


try:
    selected_feats
except NameError:
    selected_feats = all_features  

print("\nHAR will use these features：")
print(selected_feats)

# dropna
df_ml = df.dropna(subset=selected_feats + ['Y_reg']).copy()

# time split
dates = df_ml.index.get_level_values('Date')
train_mask = (dates >= pd.Timestamp('2003-01-01')) & (dates <= pd.Timestamp('2019-12-31'))
test_mask  = (dates >= pd.Timestamp('2020-01-01')) & (dates <= pd.Timestamp('2024-12-31'))

X_train = df_ml.loc[train_mask, selected_feats].to_numpy()
y_train = df_ml.loc[train_mask, 'Y_reg'].to_numpy()
X_test  = df_ml.loc[test_mask,  selected_feats].to_numpy()
y_test  = df_ml.loc[test_mask,  'Y_reg'].to_numpy()

# train har
har_model = LinearRegression()
har_model.fit(X_train, y_train)

# predict
y_pred = har_model.predict(X_test)
test_mse = mean_squared_error(y_test, y_pred)
print(f"\n[HAR Baseline] Test MSE = {test_mse:.6f}")

# what is this? Optional 
df_test_out = df_ml.loc[test_mask].copy()
df_test_out['Y_pred_HAR'] = y_pred
df_test_out.rename(columns={'Y_reg': 'Y_true'}, inplace=True)

print("\nSample predictions on test (head):")
print(df_test_out[['Y_true', 'Y_pred_HAR']].head())






HAR will use these features：
['rv_ma5', 'bpv', 'good_lag1', 'bad_lag1', 'rq', 'vix']

[HAR Baseline] Test MSE = 16.007118

Sample predictions on test (head):
                    Y_true  Y_pred_HAR
Date       Stock                      
2020-01-02 AAPL   1.898643    0.652546
           AMGN   0.995035    0.601105
           AMZN   1.372380    0.738684
           AXP    0.878470    0.563066
           BA     1.003243    0.830361


In [107]:
from sklearn.ensemble import IsolationForest

# --- 3.1 用 HAR 模型对全样本做预测，计算残差 ---
# 说明：selected_feats、har_model、df（含 Y_reg）、时间掩码均已在前文建立
df_full_ml = df.dropna(subset=selected_feats + ['Y_reg']).copy()
X_full = df_full_ml[selected_feats].to_numpy()
df_full_ml['har_pred'] = har_model.predict(X_full)
df_full_ml['residual'] = df_full_ml['Y_reg'] - df_full_ml['har_pred']

# 3.2 Isolation Forest
def fit_predict_isolation_forest_per_stock(g):
    dates_g = g.index.get_level_values('Date')
    train_mask_g = (dates_g >= pd.Timestamp('2003-01-01')) & (dates_g <= pd.Timestamp('2019-12-31'))
    g_train = g.loc[train_mask_g]

    # label "normal" if no data
    if g_train.empty:
        g['resid_z'] = 0.0
        g['if_label'] = 1
        return g

    # make z-score
    eps = 1e-8
    mu = g_train['residual'].mean()
    sd = g_train['residual'].std(ddof=0)
    g['resid_z'] = (g['residual'] - mu) / (sd + eps)

    # train if
    iso = IsolationForest(
        n_estimators=200,
        contamination='auto', 
        random_state=42,
        n_jobs=-1
    )
    iso.fit(g.loc[train_mask_g, ['resid_z']].to_numpy())

    g['if_label'] = iso.predict(g[['resid_z']].to_numpy())
    return g

df_if = df_full_ml.groupby(level='Stock', group_keys=False).apply(fit_predict_isolation_forest_per_stock)

# make abnormal label
df_if['is_abnormal'] = (df_if['if_label'] == -1).astype(int)
df_if['Y_class'] = df_if['is_abnormal']  

# check the %
dates_all = df_if.index.get_level_values('Date')
train_mask_all = (dates_all >= pd.Timestamp('2003-01-01')) & (dates_all <= pd.Timestamp('2019-12-31'))
test_mask_all  = (dates_all >= pd.Timestamp('2020-01-01')) & (dates_all <= pd.Timestamp('2024-12-31'))

train_abn_rate = df_if.loc[train_mask_all, 'is_abnormal'].mean()
test_abn_rate  = df_if.loc[test_mask_all,  'is_abnormal'].mean()
print(f"[IF] abnormal% | train: {train_abn_rate:.4%}  test: {test_abn_rate:.4%}")

print("\n example for stock 0 ）：")
sample_stock = df_if.index.get_level_values('Stock')[0]
print(df_if.xs(sample_stock, level='Stock').tail()[['Y_reg','har_pred','residual','resid_z','is_abnormal']].head())


[IF] abnormal% | train: 10.8053%  test: 17.7330%

 example for stock 0 ）：
               Y_reg  har_pred  residual   resid_z  is_abnormal
Date                                                           
2024-03-21  2.876235  2.242212  0.634023  0.305608            0
2024-03-22  5.865156  1.772104  4.093051  1.850305            1
2024-03-25  2.785550  2.120349  0.665201  0.319531            0
2024-03-26  1.776870  2.192069 -0.415199 -0.162943            0
2024-03-27  1.049129  1.484792 -0.435663 -0.172081            0
